In [1]:
import pandas as pd

# Entendimento do Problema
- A farmácia precisa entender consumo, estoque e risco de ruptura dos produtos.
- Objetivo é analisar consumo por produto, grupo e setor, identificar itens críticos e gerar indicadores para apoiar a reposição.
- Começaremos analisando a base de estoque, posteriormente a de consumo, para fazer o merge.
- Ultilizarei sempre o conceito de ETL.

## Visualização inicial

In [5]:
df_estoque = pd.read_csv("../dados/estoque_atual.csv", sep=",")
display(df_estoque.head(4))
display(df_estoque.tail(4))

,Codigo,Produto,Grupo,Estoque_Atual,Estoque_Minimo,Estoque_Maximo,Tempo_Reposicao_Dias
0,1,Dipirona 500mg/mL,Medicamento,122,122,592,13
1,2,"Soro Fisiologico 0,9% 500mL",Solucoes,91,50,864,13
2,3,Soro Glicosado 5% 500mL,Solucoes,478,117,689,6
3,4,Equipo Macrogotas,Material Hospitalar,891,181,511,4


,Codigo,Produto,Grupo,Estoque_Atual,Estoque_Minimo,Estoque_Maximo,Tempo_Reposicao_Dias
21,22,Mascara Cirurgica,Material Hospitalar,84,118,838,3
22,23,Compressa Gaze,Material Hospitalar,667,92,430,19
23,24,Esparadrapo,Material Hospitalar,411,192,810,3
24,25,Alcool 70%,Material Hospitalar,398,34,723,9


In [6]:
display(df_estoque.info())
display(df_estoque.describe())
display(df_estoque.columns.to_list())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Codigo                25 non-null     int64 
 1   Produto               25 non-null     object
 2   Grupo                 25 non-null     object
 3   Estoque_Atual         25 non-null     int64 
 4   Estoque_Minimo        25 non-null     int64 
 5   Estoque_Maximo        25 non-null     int64 
 6   Tempo_Reposicao_Dias  25 non-null     int64 
dtypes: int64(5), object(2)
memory usage: 1.5+ KB


None

,Codigo,Estoque_Atual,Estoque_Minimo,Estoque_Maximo,Tempo_Reposicao_Dias
count,25.000000,25.000000,25.000000,25.000000,25.000000
mean,13.000000,435.080000,96.840000,650.840000,10.840000
std,7.359801,271.175976,54.558592,206.642743,5.080026
min,1.000000,60.000000,31.000000,271.000000,3.000000
25%,7.000000,186.000000,47.000000,511.000000,6.000000
50%,13.000000,411.000000,83.000000,687.000000,12.000000
75%,19.000000,504.000000,121.000000,810.000000,15.000000
max,25.000000,977.000000,193.000000,993.000000,19.000000


['Codigo',
 'Produto',
 'Grupo',
 'Estoque_Atual',
 'Estoque_Minimo',
 'Estoque_Maximo',
 'Tempo_Reposicao_Dias']

## Verificação de nulo e duplicados

In [8]:
display(df_estoque.isna().sum())
display(df_estoque.duplicated().sum())

Codigo                  0
Produto                 0
Grupo                   0
Estoque_Atual           0
Estoque_Minimo          0
Estoque_Maximo          0
Tempo_Reposicao_Dias    0
dtype: int64

np.int64(0)

In [10]:
display(df_estoque["Grupo"].unique())

array(['Medicamento', 'Solucoes', 'Material Hospitalar', 'Antibiotico'],
      dtype=object)

## Análise inicial da base

- Quantos produtos e grupos existem?

In [21]:
display(df_estoque["Produto"].unique())
display(df_estoque["Grupo"].unique())

array(['Dipirona 500mg/mL', 'Soro Fisiologico 0,9% 500mL',
       'Soro Glicosado 5% 500mL', 'Equipo Macrogotas', 'Seringa 10mL',
       'Seringa 20mL', 'Agulha 25x7', 'Luva Procedimento M',
       'Omeprazol 40mg', 'Ceftriaxona 1g', 'Ampicilina 1g',
       'Metoclopramida 10mg', 'Ondansetrona 4mg', 'Furosemida 20mg',
       'Hidrocortisona 100mg', 'Adrenalina 1mg/mL', 'Atropina 0,25mg/mL',
       'Cloreto de Sodio 20%', 'Glicose 50%', 'Cateter Venoso 20G',
       'Cateter Venoso 22G', 'Mascara Cirurgica', 'Compressa Gaze',
       'Esparadrapo', 'Alcool 70%'], dtype=object)

array(['Medicamento', 'Solucoes', 'Material Hospitalar', 'Antibiotico'],
      dtype=object)

- Qual grupo tem mais produtos?

In [29]:
df_produto_grupo = df_estoque.groupby("Grupo").agg(
    Total_Produtos=("Produto", "count")
).reset_index().sort_values(by="Total_Produtos", ascending=False)
display(df_produto_grupo)

,Grupo,Total_Produtos
1,Material Hospitalar,11
2,Medicamento,8
3,Solucoes,4
0,Antibiotico,2
